# PPG v0.1 reference calculation

This is the explanatory oracle for the Portfolio–Paycheck Gap Index, not the production pipeline. It uses nine pinned monthly observations to calculate 1980 Q1–Q3. Every input is local, every transformation remains inspectable, and only the Python standard library is required.


## 1. Load pinned inputs

JKP `ret` is a decimal monthly US-dollar excess return. FRED `TB3MS` is a monthly-average annual discount yield in percent. BLS `value` is a seasonally adjusted median weekly paycheck in current dollars. Dates are labels; no interpolation or forward fill is allowed.


In [ ]:
from calendar import monthrange
import csv
import io
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(path for path in candidates if (path / 'cairn.toml').exists())

SOURCE_DIR = PROJECT_ROOT / 'tests' / 'fixtures' / 'sources'
OUTPUT_DIR = PROJECT_ROOT / 'build' / 'reference'

def read_csv(path):
    with path.open(newline='', encoding='utf-8') as handle:
        return list(csv.DictReader(handle))

market_source = read_csv(SOURCE_DIR / 'jkp-usa-mkt-monthly-vw.csv')
treasury_source = read_csv(SOURCE_DIR / 'fred-tb3ms.csv')
paycheck_source = read_csv(SOURCE_DIR / 'bls-les1252881500.csv')
print(f'loaded {len(market_source)} market, {len(treasury_source)} Treasury, and {len(paycheck_source)} paycheck rows')


## 2. Reconstruct monthly total return

For a 91-day bill, discount yield `y` implies price `1 - (y/100) × 91/360`. We geometrically accrue that discount over the month's calendar days to obtain a decimal Treasury-return proxy, then add it to JKP's decimal excess return. Monthly market wealth starts at an arbitrary 1 immediately before January; rebasing later removes this arbitrary level.


In [ ]:
reference_months = [f'1980-{month:02d}' for month in range(1, 10)]
excess_by_month = {row['date'][:7]: float(row['ret']) for row in market_source}
yield_by_month = {row['observation_date'][:7]: float(row['TB3MS']) for row in treasury_source}
assert set(reference_months) <= excess_by_month.keys()
assert set(reference_months) <= yield_by_month.keys()

monthly_rows = []
market_wealth = 1.0
for month_label in reference_months:
    year, month = map(int, month_label.split('-'))
    days = monthrange(year, month)[1]
    excess_return = excess_by_month[month_label]
    annual_discount_yield = yield_by_month[month_label]
    bill_price = 1.0 - (annual_discount_yield / 100.0) * (91.0 / 360.0)
    assert 0.0 < bill_price <= 1.0
    treasury_return = bill_price ** (-days / 91.0) - 1.0
    market_return = excess_return + treasury_return
    assert market_return > -1.0
    market_wealth *= 1.0 + market_return
    monthly_rows.append({
        'month': month_label,
        'excess_return': excess_return,
        'annual_discount_yield_percent': annual_discount_yield,
        'calendar_days': days,
        'bill_price': bill_price,
        'treasury_return': treasury_return,
        'market_return': market_return,
        'market_wealth': market_wealth,
    })

for row in monthly_rows:
    print(row['month'], f"rf={row['treasury_return']:.12f}", f"market={row['market_return']:.12f}", f"wealth={row['market_wealth']:.12f}")


## 3. Align quarter-end market wealth with quarterly paycheck dollars

The market value is the cumulative wealth after March, June, or September. The paycheck is already a quarterly BLS observation. A quarter survives only when both values exist, are finite, and are strictly positive. The intermediate values remain in `quarterly_rows`.


In [ ]:
paycheck_by_quarter = {
    f"{row['year']}-Q{int(row['period'][1:])}": float(row['value'])
    for row in paycheck_source
    if row['value'] != '-'
}
quarterly_rows = []
for row in monthly_rows:
    month = int(row['month'][-2:])
    if month % 3:
        continue
    quarter = f"{row['month'][:4]}-Q{month // 3}"
    paycheck_value = paycheck_by_quarter[quarter]
    assert row['market_wealth'] > 0.0 and paycheck_value > 0.0
    quarterly_rows.append({
        'quarter': quarter,
        'market_value': row['market_wealth'],
        'paycheck_value': paycheck_value,
    })
print(quarterly_rows)


## 4. Rebase both components and calculate PPG

1980 Q1 equals 100 for both components. `MARKET = 100 × M/M_base`; `PAYCHECK = 100 × W/W_base`; and `PPG = 100 × MARKET/PAYCHECK`. Inputs and intermediates remain unrounded. Six-decimal formatting happens only when the fixture is serialized.


In [ ]:
base = next(row for row in quarterly_rows if row['quarter'] == '1980-Q1')
for row in quarterly_rows:
    row['market_component'] = 100.0 * row['market_value'] / base['market_value']
    row['paycheck_component'] = 100.0 * row['paycheck_value'] / base['paycheck_value']
    row['ppg'] = 100.0 * row['market_component'] / row['paycheck_component']
    direct = 100.0 * (row['market_value'] / base['market_value']) / (row['paycheck_value'] / base['paycheck_value'])
    assert abs(row['ppg'] - direct) <= 1e-12
print([(row['quarter'], row['market_component'], row['paycheck_component'], row['ppg']) for row in quarterly_rows])


## 5. Export and compare the compact parity fixture

The exported CSV retains quarter-end market wealth (arbitrary units), weekly paycheck dollars, and all three unitless index levels. Comparing exact bytes to the independently hand-calculated fixture makes this notebook useful as a later production parity oracle.


In [ ]:
columns = ['quarter', 'market_value', 'paycheck_value', 'market_component', 'paycheck_component', 'ppg']
buffer = io.StringIO(newline='')
writer = csv.DictWriter(buffer, fieldnames=columns, lineterminator='\n')
writer.writeheader()
for row in quarterly_rows:
    writer.writerow({'quarter': row['quarter'], **{column: f"{row[column]:.6f}" for column in columns[1:]}})
reference_csv = buffer.getvalue()
expected = (PROJECT_ROOT / 'tests' / 'fixtures' / 'reference-calculation.csv').read_text(encoding='utf-8')
assert reference_csv == expected
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'reference-calculation.csv').write_text(reference_csv, encoding='utf-8', newline='')
print(reference_csv)


## 6. Plot the retained components and headline

All three lines are unitless indexes with 1980 Q1 = 100. Showing the components prevents a PPG move from being mistaken for a one-sided claim about either stocks or pay. The SVG is generated directly so the clean reference environment needs no plotting package.


In [ ]:
width, height = 720, 400
left, right, top, bottom = 70, 24, 38, 58
series = [
    ('MARKET', '#2563eb', [row['market_component'] for row in quarterly_rows]),
    ('PAYCHECK', '#d97706', [row['paycheck_component'] for row in quarterly_rows]),
    ('PPG', '#111827', [row['ppg'] for row in quarterly_rows]),
]
values = [value for _, _, points in series for value in points]
y_min = 95.0
y_max = max(values) + 5.0
x = lambda index: left + index * (width - left - right) / (len(quarterly_rows) - 1)
y = lambda value: top + (y_max - value) * (height - top - bottom) / (y_max - y_min)
svg = [f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">']
svg += ['<rect width="100%" height="100%" fill="white"/>', '<style>text{font-family:system-ui,sans-serif;fill:#374151}.title{font-size:18px;font-weight:600}.tick{font-size:12px}.legend{font-size:13px;font-weight:600}</style>', '<text class="title" x="70" y="24">PPG reference calculation · 1980 Q1 = 100</text>']
for tick in range(100, int(y_max) + 1, 5):
    svg.append(f'<line x1="{left}" x2="{width-right}" y1="{y(tick):.2f}" y2="{y(tick):.2f}" stroke="#e5e7eb"/>')
    svg.append(f'<text class="tick" x="{left-9}" y="{y(tick)+4:.2f}" text-anchor="end">{tick}</text>')
for index, row in enumerate(quarterly_rows):
    svg.append(f'<text class="tick" x="{x(index):.2f}" y="{height-bottom+24}" text-anchor="middle">{row["quarter"]}</text>')
for legend_index, (name, color, points) in enumerate(series):
    coordinates = ' '.join(f'{x(index):.2f},{y(value):.2f}' for index, value in enumerate(points))
    svg.append(f'<polyline points="{coordinates}" fill="none" stroke="{color}" stroke-width="3"/>')
    legend_x = left + 180 * legend_index
    svg.append(f'<line x1="{legend_x}" x2="{legend_x+26}" y1="{height-17}" y2="{height-17}" stroke="{color}" stroke-width="3"/>')
    svg.append(f'<text class="legend" x="{legend_x+34}" y="{height-12}">{name}</text>')
svg.append('</svg>')
plot_path = OUTPUT_DIR / 'ppg-reference.svg'
plot_path.write_text('\n'.join(svg) + '\n', encoding='utf-8')
print(f'wrote {plot_path.relative_to(PROJECT_ROOT)}')


The reference rises to 114.258 in Q2 and 124.847 in Q3 because reconstructed market wealth grew faster than the median paycheck over those two quarters. This tiny window validates mechanics only; it is not an economic conclusion or a claim that the base is equilibrium.
